# Graph + FGW: few-shot and zero-shot classification

Each augmented transcript is a graph (one node per cue, plus the start node) compared with the Fused Gromov-Wasserstein distance. Ability and benevolence are evaluated separately.

* **Few-shot** ($m$ labeled examples per level): one FGW barycenter per level is computed from its $m$ examples, and every transcript gets the level of the nearest barycenter. For each fold, the $m$ examples per level come from the other folds, so a transcript is never classified with a barycenter it contributed to.
* **Zero-shot** (unsupervised): $k$-means with $K = 3$ where the distance is FGW and the centroids are FGW barycenters. Each run starts from $K$ transcripts drawn at random; the run with the lowest inertia (sum of distances to the assigned centroid) is kept. Each cluster is mapped to a level with the Hungarian algorithm.

Barycenters have as many nodes as the median of their graphs. The folds, examples and scores are the same as for the other methods (`lib/evaluation.py`).

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, agreement_table, evaluate, load_or_create_split, load_predictions,
                        load_ratings, load_transcripts, map_clusters, plot_confusion, summarize,
                        zero_shot_subset)
from graph_multimodalsentence import Graph, compute_barycenter
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
SPLIT_DIR = "splits"            # folds and few-shot examples, shared by all methods
OUTPUT_DIR = "results"

DIMENSIONS = ["ability", "benevolence"]
REFERENCE = "instructed"        # "instructed" or "perceived"
# Individual ratings of the participants, needed for REFERENCE = "perceived": csv with one row
# per rating and columns id (<condition>#<row>), dimension, rater, rating (Low / Medium / High).
# The perceived level is the median of the ratings; only rated transcripts are evaluated.
RATINGS_FILE = None

M = 10                          # labeled examples per level (few-shot)
N_FOLDS = 5
SEED = 0
N_PER_LEVEL = None              # evaluate at most this many transcripts per level (None: all)
ZERO_SHOT_N_PER_LEVEL = 100     # transcripts per level clustered in the zero-shot setting (None: all)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
BARYCENTER_MAX_ITER = 50        # iterations of the barycenter computation
N_KMEANS_INIT = 5               # k-means restarts
KMEANS_MAX_ITER = 20            # k-means iterations per restart

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data

All `full_*` conditions. Each transcript has an `id` = `<condition>#<row>`. Folds and few-shot examples are loaded from `splits/` (created by the first notebook run, whichever method it is), so every method uses the same ones. **Keep the data parameters identical across notebooks.**

In [ ]:
df = load_transcripts(DATA_DIR, DIMENSIONS, REFERENCE, RATINGS_FILE, N_PER_LEVEL, SEED)
splits = {dimension: load_or_create_split(df[df["dimension"] == dimension], dimension, REFERENCE,
                                          SPLIT_DIR, M, N_FOLDS, SEED)
          for dimension in DIMENSIONS}
df.groupby(["dimension", "level"]).size().unstack()

## Graphs

Cue descriptions are embedded once (the same descriptions occur in many transcripts).

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))
start = time.time()
texts = df.drop_duplicates("id").set_index("id")["text"]
graphs = {i: Graph.from_sentence(text, embedding) for i, text in texts.items()}
print(f"{len(graphs)} graphs built in {time.time() - start:.0f} s")

In [ ]:
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")


def distances_to(ids, centers):
    # FGW distance between the graphs of ids and each center, shape (len(ids), len(centers)).
    return np.array([[fgw.graph_d(graphs[i], center) for center in centers] for i in ids])


def barycenter(ids):
    return compute_barycenter([graphs[i] for i in ids], alpha=ALPHA, max_iter=BARYCENTER_MAX_ITER)

## Few-shot: nearest FGW barycenter

In [ ]:
def few_shot(folds, examples):
    # Predicted level of every transcript, from the barycenters of its own fold.
    predictions = []
    for fold in range(N_FOLDS):
        start = time.time()
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        barycenters = [barycenter(support.loc[support["level"] == level, "id"]) for level in LEVELS]
        distances = distances_to(test["id"], barycenters)
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
        print(f"  fold {fold}: {len(test)} transcripts, {time.time() - start:.0f} s")
    return pd.concat(predictions, ignore_index=True)

## Zero-shot: FGW $k$-means and Hungarian mapping

In [ ]:
def fgw_kmeans(ids, k, rng):
    # Returns the cluster of each graph and the inertia of the best restart.
    ids = list(ids)
    best_labels, best_inertia = None, np.inf
    for restart in range(N_KMEANS_INIT):
        centers = [graphs[i] for i in rng.choice(ids, k, replace=False)]
        labels = None
        for iteration in range(KMEANS_MAX_ITER):
            distances = distances_to(ids, centers)
            new_labels = distances.argmin(axis=1)
            if labels is not None and np.array_equal(new_labels, labels):
                break
            labels = new_labels
            for c in range(k):
                members = [i for i, label in zip(ids, labels) if label == c]
                if not members:  # empty cluster: restart it from the graph farthest from its center
                    members = [ids[distances.min(axis=1).argmax()]]
                centers[c] = barycenter(members)
        distances = distances_to(ids, centers)
        labels = distances.argmin(axis=1)
        inertia = distances[np.arange(len(ids)), labels].sum()
        print(f"  restart {restart}: {iteration + 1} iterations, inertia {inertia:.4f}")
        if inertia < best_inertia:
            best_labels, best_inertia = labels, inertia
    return best_labels, best_inertia


def zero_shot(data):
    clusters, _ = fgw_kmeans(data["id"], len(LEVELS), np.random.default_rng(SEED))
    return pd.DataFrame({"id": data["id"], "true": data["level"].to_numpy(), "cluster": clusters,
                         "pred": map_clusters(clusters, data["level"])})

## Run

In [ ]:
predictions = {}
for dimension in DIMENSIONS:
    data = df[df["dimension"] == dimension]
    print(f"{dimension}, few-shot")
    predictions[(dimension, "few-shot")] = few_shot(*splits[dimension])
    print(f"{dimension}, zero-shot")
    predictions[(dimension, "zero-shot")] = zero_shot(zero_shot_subset(data, ZERO_SHOT_N_PER_LEVEL, SEED))

## Results

Accuracy, macro-F1 and agreement with the reference: quadratic-weighted Cohen's $\kappa$ (levels are ordered, so predicting High for Low counts more than predicting Medium), with a 95% percentile bootstrap confidence interval over transcripts (1000 resamples).

Few-shot: mean ± standard deviation over folds. Chance level is 1/3 for accuracy and 0 for $\kappa$.

In [ ]:
results = evaluate(predictions, method="graph + FGW", reference=REFERENCE)
results.to_csv(os.path.join(OUTPUT_DIR, f"fgw_results_{REFERENCE}.csv"), index=False)
for (dimension, setting), pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"fgw_predictions_{dimension}_{setting}_{REFERENCE}.csv"),
                index=False)
summarize(results)

All folds pooled; zero-shot also with ARI and AMI, which do not depend on the cluster-to-level mapping.

In [ ]:
pooled = results[results["fold"] == "all"].drop(columns="fold").round(3)
pooled.insert(pooled.columns.get_loc("kappa") + 1, "kappa [95% CI]",
              [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
               zip(pooled["kappa"], pooled["kappa_ci_low"], pooled["kappa_ci_high"])])
pooled

In [ ]:
plot_confusion(predictions, REFERENCE, os.path.join(OUTPUT_DIR, f"fgw_confusion_{REFERENCE}.png"))
plt.show()

## Comparison with the embedding baseline

If `embedding_baseline.ipynb` was run with the same reference.

In [ ]:
path = os.path.join(OUTPUT_DIR, f"embedding_results_{REFERENCE}.csv")
if os.path.exists(path):
    both = pd.concat([results, pd.read_csv(path)])
    both = both[both["fold"].astype(str) == "all"]
    display(both.pivot_table(index=["dimension", "setting"], columns="method",
                             values=["accuracy", "macro_f1", "kappa"]).round(3))

## Agreement with human raters

AGREEMENT (quadratic-weighted Cohen's $\kappa$, 95% CI), only with `REFERENCE = "perceived"`.

* **Human raters vs human**: each rating compared with the median of the other raters of the same transcript (leave-one-rater-out). This is the agreement a single human reaches, i.e. the ceiling for the methods. Its CI resamples transcripts with all their ratings.
* **Method vs human**: the level predicted by the method compared with the perceived level (median of all raters). Its CI resamples transcripts.

In [ ]:
if REFERENCE == "perceived":
    ratings = load_ratings(RATINGS_FILE)
    methods = {"Graph + FGW": predictions}
    embedding_predictions = load_predictions(OUTPUT_DIR, "embedding", REFERENCE, DIMENSIONS)
    if embedding_predictions:  # run embedding_baseline.ipynb first to include it
        methods["Embeddings"] = embedding_predictions
    agreement = agreement_table(ratings, methods, DIMENSIONS)
    agreement.to_csv(os.path.join(OUTPUT_DIR, "agreement_fgw.csv"))
    display(agreement)
else:
    print("Set REFERENCE = \"perceived\" and RATINGS_FILE to compare with the human raters.")